Trabalho de Data Science
Alunos:
    Leonardo Lopes Oliveira
    Felipe Krzyzanovski dos Santos Menezes

In [ ]:
!pip install pandas numpy plotly scikit-learn

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

Analise do csv de cybersecurity

In [ ]:
df = pd.read_csv('cybersecurity.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.groupby(df["timestamp"].dt.date).size().reset_index(name="attacks")
fig = px.line(
    df,
    x="timestamp",
    y="attacks",
    title="Number of Attacks Over Time"
)

fig.show()


A linha mostra o número de ataques por dia entre início de outubro e final de dezembro de 2025. O comportamento dominante é relativamente estável, os valores ficam majoritariamente entre 100 e 125 ataques por dia. Isso sugere um tráfego malicioso constante — algo extremamente comum na internet pública. Bots fazem varreduras contínuas procurando portas abertas e vulnerabilidades.

Analise motoristas dormindo

In [ ]:
df = pd.read_csv('Driver_Drowsiness_3000.csv')
X = df[['EAR', 'MAR']]
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LogisticRegression()
modelo.fit(X_train, y_train)

x_min, x_max = X['EAR'].min() - 0.05, X['EAR'].max() + 0.05
y_min, y_max = X['MAR'].min() - 0.05, X['MAR'].max() + 0.05

eixo_x = np.arange(x_min, x_max, 0.01)
eixo_y = np.arange(y_min, y_max, 0.01)
xx, yy = np.meshgrid(eixo_x, eixo_y)

Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)


In [ ]:
fig1 = go.Figure()

fig1.add_trace(go.Contour(
    x=eixo_x, y=eixo_y, z=Z,
    colorscale='RdBu', 
    opacity=0.3,
    showscale=False,
    hoverinfo='skip'
))

df_acordado = df[df['Target'] == 0]
fig1.add_trace(go.Scatter(
    x=df_acordado['EAR'], y=df_acordado['MAR'],
    mode='markers', name='Acordado (0)',
    marker=dict(color='blue', line=dict(color='white', width=1))
))

df_sonolento = df[df['Target'] == 1]
fig1.add_trace(go.Scatter(
    x=df_sonolento['EAR'], y=df_sonolento['MAR'],
    mode='markers', name='Sonolento (1)',
    marker=dict(color='red', line=dict(color='white', width=1))
))

fig1.update_layout(
    title='Fronteira de Decisão - Regressão Logística',
    xaxis_title='EAR (Fechamento do Olho)',
    yaxis_title='MAR (Abertura da Boca)',
    width=800, height=600
)

fig1.show()


In [ ]:
y_pred = modelo.predict(X_test)
matriz = confusion_matrix(y_test, y_pred)

labels = ['Acordado (0)', 'Sonolento (1)']

fig2 = px.imshow(
    matriz, 
    text_auto=True, 
    color_continuous_scale='Blues',
    x=labels, 
    y=labels,
    labels=dict(x="Previsão do Modelo", y="Realidade")
)

fig2.update_layout(title='Matriz de Confusão', width=600, height=500)
fig2.show()

O primeiro gráfico apresenta a fronteira de decisão do modelo. Nesse gráfico, cada ponto representa uma amostra do conjunto de dados, em que os pontos azuis correspondem a motoristas classificados como acordados e os pontos vermelhos correspondem a motoristas classificados como sonolentos. A área colorida representa a região de decisão definida pelo modelo de regressão logística, indicando como os valores de EAR e MAR são utilizados para separar as duas classes. Observa-se que as amostras estão bem distribuídas em regiões distintas, demonstrando que essas duas variáveis são eficazes para diferenciar os estados analisados.

O segundo gráfico corresponde à matriz de confusão, utilizada para avaliar o desempenho do modelo no conjunto de teste. Essa matriz compara as previsões realizadas pelo modelo com os valores reais presentes nos dados. Os resultados indicam 287 classificações corretas para a classe “acordado” e 313 classificações corretas para a classe “sonolento”, não havendo ocorrências de classificações incorretas. Esses resultados indicam que, para o conjunto de dados analisado, o modelo apresentou um desempenho elevado na distinção entre os dois estados do motorista.

bitcoin analise

In [ ]:
df_btc = pd.read_csv('bitcoin.csv')
df_btc['Date'] = pd.to_datetime(df_btc['Date'])
df_btc = df_btc.sort_values('Date')

fig = px.line(
    df_btc, 
    x='Date', 
    y='Close', 
    title='Histórico de Preço do Bitcoin (USD)',
    labels={'Date': 'Data', 'Close': 'Preço em Dólar (USD)'},
    log_y=True,
    template='plotly_dark' # Um tema escuro fica bem legal para dados financeiros
)

fig.update_traces(line=dict(width=1.5, color='#f2a900'))

fig.update_layout(
    yaxis=dict(
        tickformat="$",
        gridcolor='rgba(255, 255, 255, 0.1)'
    ),
    xaxis=dict(gridcolor='rgba(255, 255, 255, 0.1)'),
    hovermode="x unified"
)

fig.update_xaxes(rangeslider_visible=True)

fig.show()

Nesta análise, utilizei um conjunto de dados contendo o histórico de preços do Bitcoin. Inicialmente carreguei o arquivo bitcoin.csv, converti a coluna de datas para o formato datetime e organizei os dados em ordem cronológica para garantir a correta visualização da evolução do preço ao longo do tempo.

O gráfico apresentado representa uma série temporal do preço de fechamento do Bitcoin em dólar (USD). No eixo horizontal está a data, enquanto no eixo vertical está o preço do Bitcoin. Para melhorar a visualização das variações ao longo dos anos, utilizei uma escala logarítmica no eixo Y (log_y=True). Esse tipo de escala é comum em análises financeiras porque permite observar variações percentuais de forma mais clara, principalmente quando os valores variam de centavos para dezenas de milhares de dólares.

A partir do gráfico, é possível observar a forte valorização do Bitcoin ao longo do tempo. Nos primeiros anos, entre aproximadamente 2010 e 2012, o ativo possuía valores muito baixos, inferiores a alguns dólares. A partir de 2013, observa-se um crescimento mais expressivo, indicando o início de maior interesse e adoção da criptomoeda.

Entre 2017 e 2018, o gráfico mostra um aumento significativo no preço, seguido por uma queda posterior, caracterizando um ciclo típico de valorização e correção do mercado. Nos anos seguintes, especialmente entre 2020 e 2021, ocorre novamente um período de forte valorização, no qual o Bitcoin atinge valores muito mais elevados em comparação aos anos anteriores.

Nos períodos mais recentes, o gráfico mostra oscilações de preço, indicando que o mercado de criptomoedas apresenta alta volatilidade. Essas variações são comuns em ativos digitais, pois o preço é fortemente influenciado por fatores como adoção tecnológica, especulação de mercado, políticas econômicas e eventos globais.

Além disso, o gráfico inclui um range slider, que permite explorar períodos específicos da série temporal, facilitando a análise de determinados intervalos de tempo.

De forma geral, essa visualização permite compreender a evolução histórica do Bitcoin, identificar períodos de crescimento acelerado e observar momentos de correção de mercado, fornecendo uma visão clara do comportamento desse ativo ao longo dos anos.